# IMU acceleration — last minutes of observing

Plots the elevation-stage IMU accelerometer (`imu_el_accel_x/y/z`) against time
for the final `MINUTES` of the marjum-2026-07 campaign, read straight from the
per-spectrum metadata in `marjum-2026-07/data/` via `MetadataIndex`.

Time is `time_best`. The campaign's last *sample* is 2026-07-18 03:08:49 UTC;
the "03:22" often quoted is a filename (file close time) artifact. Rows where
the IMU reported no reading are NaN and simply don't plot.

In [ ]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PLOT_BACKEND = os.environ.get('EIGSEP_NOTEBOOK_BACKEND', 'widget')
get_ipython().run_line_magic('matplotlib', PLOT_BACKEND)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'marjum-2026-07/data').is_dir())
CAMPAIGN = ROOT / 'marjum-2026-07'

import eigsep_data
from eigsep_data import MetadataIndex
eigsep_data.set_campaign_root(CAMPAIGN)

MINUTES = 20

In [ ]:
meta = MetadataIndex(CAMPAIGN / 'data').select().meta
t = meta.time_best.astype(float)
last = meta[t >= t.max() - MINUTES * 60].sort_values('time_best')

times = pd.to_datetime(last.time_best, unit='s', utc=True)
print('window: %s -> %s UTC' % (times.min().strftime('%Y-%m-%d %H:%M:%S'),
                                times.max().strftime('%H:%M:%S')))
print('%d rows from %d files; IMU present on %d'
      % (len(last), last.file.nunique(), last.imu_el_accel_x.notna().sum()))

In [ ]:
plt.rcParams.update({'font.size': 14, 'axes.spines.top': False,
                     'axes.spines.right': False})

fig, axes = plt.subplots(1, 1, figsize=(8, 8), constrained_layout=True)
axes = [axes, axes, axes]
for axis, comp, color in zip(axes, 'xyz', ['tab:blue', 'tab:orange', 'tab:green']):
    axis.plot(times, last[f'imu_el_accel_{comp}'], '-', ms=3, color=color)
    axis.set_ylabel(f'accel {comp} [m/s$^2$]')
axes[-1].set_xlabel('Time [UTC]')
axes[0].set_title('Elevation-stage IMU, last %d minutes of observing' % MINUTES)
fig.autofmt_xdate()
plt.show()